# HAR-RV exploration (model_3, `har_rv`)

Steps 1–3 of the roadmap: build the continuous calendar, features, targets; check them against `actuals.csv`; then in-sample HAR on train.

## Step 1 — Calendar, features, targets

Setup: run from the repo root so `pipeline` imports and data paths resolve.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "model_3":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from pipeline.csv_io import DAILY, ACTUALS, HORIZONS, load

root = Path("data/alpaca")
phases = ("train", "validation", "test")

One continuous calendar across all three splits, so rolling windows and shifts carry across split boundaries.

In [ ]:
daily = pd.concat([load(root / p / "daily_inputs.csv", DAILY) for p in phases],
                  ignore_index=True).set_index("date")
assert daily.index.is_unique and daily.index.is_monotonic_increasing
rv = daily["spy_rv"]
print(len(daily), daily.index[0], daily.index[-1])   # expect 2733 rows

HAR features at origin *t* — all trailing windows **include** *t* (available at the 17:00 cutoff).

In [ ]:
feats = pd.DataFrame({
    "rv_d": rv,
    "rv_w": rv.rolling(5).mean(),
    "rv_m": rv.rolling(22).mean(),
})
feats.describe()

Targets: mean of the **next** *h* sessions (excludes *t*). Baseline: mean of the last *h* sessions **including** *t*.

In [ ]:
target   = {h: rv.rolling(h).mean().shift(-h) for h in HORIZONS}
baseline = {h: rv.rolling(h).mean()           for h in HORIZONS}

Check against `actuals.csv`. Expect max diffs ~1e-19 (float noise) and blank mismatches only in **train** (28 rows at the end of 2020, where the continuous calendar can see 2021 — don't use those labels for train-only fitting).

In [ ]:
actuals = pd.concat([load(root / p / "actuals.csv", ACTUALS).assign(phase=p) for p in phases],
                    ignore_index=True)
built = pd.concat(
    {h: pd.DataFrame({"actual_mine": target[h], "baseline_mine": baseline[h]}) for h in HORIZONS},
    names=["horizon", "origin_date"],
).reset_index()
cmp = actuals.merge(built, on=["origin_date", "horizon"], how="left")

for col in ("actual", "baseline"):
    cmp[f"{col}_diff"]  = (cmp[col] - cmp[f"{col}_mine"]).abs()
    cmp[f"{col}_blank"] = cmp[col].isna() != cmp[f"{col}_mine"].isna()

cmp.groupby(["phase", "horizon"]).agg(
    actual_maxdiff=("actual_diff", "max"),   actual_blank_mismatch=("actual_blank", "sum"),
    base_maxdiff=("baseline_diff", "max"),   base_blank_mismatch=("baseline_blank", "sum"),
)

## Step 2 — Export persistence as a forecast; it must score exactly like the built-in Baseline

Goes through the same path HAR will use (schedule → `export` → report), so this proves the plumbing. Written to `reports/preview/` (gitignored) — a check, not a submission.

`generate_report` is the function behind `python -m pipeline.report`.

In [ ]:
from pipeline.csv_io import FORECAST, KEY, export
from pipeline.report import generate_report

check_dir = Path("reports/preview/har_rv_checks")

for phase in ("validation", "test"):
    sched_path = root / phase / "forecast_schedule.csv"
    schedule = load(sched_path, KEY)
    sub = schedule.assign(
        model_id="persistence_check",
        prediction=[baseline[h].get(day) for day, h in schedule.itertuples(index=False, name=None)],
    )
    path = check_dir / phase / "persistence_check.csv"
    export(sub[list(FORECAST)], path, FORECAST, schedule=schedule)

    scores, coverage = generate_report(sched_path, root / phase / "actuals.csv", [path],
                                       "persistence_check", check_dir / phase)
    print(f"--- {phase} ---")
    display(scores)
    display(coverage.pivot(index="method", columns="horizon", values=["scored", "missing_predictions"]))

    rmse = scores.set_index("method").filter(like="rmse")
    assert (rmse.loc["persistence_check"] == rmse.loc["Baseline"]).all(), "scores differ!"

## Step 3 — In-sample HAR on train

Direct HAR, one OLS per horizon (Corsi 2009):

$$Y_{t,h} = \beta_0 + \beta_d\,RV_t + \beta_w\,RV^{(5)}_t + \beta_m\,RV^{(22)}_t + \varepsilon_{t,h}$$

- **Sample:** train origins with full 22-day features and a label that matured *inside* train (`pos + h <= last train pos`) — drops the 28 end-of-2020 rows that peek into 2021.
- **Standard errors:** for h = 5, 22 consecutive targets share days, so residuals are autocorrelated → Newey–West (HAC) with `maxlags = h`. Only affects t-stats, not the betas.
- **Sensitivity:** refit on labels matured by 2020-01-31 (pre-COVID) to see how much March 2020 moves the betas.
- In-sample only — this is for understanding and slides, not model selection.

In [ ]:
import numpy as np
import statsmodels.api as sm   # model_3/requirements.txt

n_train = len(load(root / "train" / "daily_inputs.csv", DAILY))
pos = pd.Series(np.arange(len(daily)), index=daily.index)   # date -> position in full calendar
X_all = sm.add_constant(feats)

def train_sample(h, last_date=None):
    """Rows with complete features and a label matured by last_date (default: last train session)."""
    end_pos = n_train - 1 if last_date is None else pos.loc[:last_date].iloc[-1]
    ok = feats.notna().all(axis=1) & target[h].notna() & (pos + h <= end_pos)
    return X_all[ok], target[h][ok]

def fit_har(h, last_date=None):
    X, y = train_sample(h, last_date)
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": h})

names = {"const": "b0", "rv_d": "b_d", "rv_w": "b_w", "rv_m": "b_m"}

def betas_table(fits):
    return pd.DataFrame({
        h: pd.concat([f.params.rename(names),
                      f.tvalues.rename(lambda k: "t_" + names[k]),
                      pd.Series({"slope_sum": f.params.iloc[1:].sum(), "R2": f.rsquared, "n": f.nobs})])
        for h, f in fits.items()
    }).T.rename_axis("horizon")

fits     = {h: fit_har(h) for h in HORIZONS}                 # full train
fits_pre = {h: fit_har(h, "2020-01-31") for h in HORIZONS}   # pre-COVID

with pd.option_context("display.float_format", "{:.3g}".format):
    print("Full train 2016–2020"); display(betas_table(fits))
    print("Pre-COVID (labels matured by 2020-01-31)"); display(betas_table(fits_pre))

In [ ]:
# In-sample RMSE vs persistence on the same rows, plus the largest residuals (h = 1)
def rmse_of(e):
    return float(np.sqrt(np.mean(e ** 2)))

rows = []
for h, f in fits.items():
    X, y = train_sample(h)
    rows.append({"horizon": h, "n": len(y),
                 "rmse_har": rmse_of(y - f.fittedvalues),
                 "rmse_persistence": rmse_of(y - baseline[h][y.index]),
                 "fitted_nonpositive": int((f.fittedvalues <= 0).sum())})
display(pd.DataFrame(rows))
fits[1].resid.abs().nlargest(8)

In [ ]:
# Actual vs fitted, log scale (variance spans ~3 orders of magnitude)
import matplotlib.pyplot as plt

ACTUAL, FITTED = "#52514e", "#2a78d6"
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, layout="constrained")
for ax, (h, f) in zip(axes, fits.items()):
    X, y = train_sample(h)
    t = pd.to_datetime(y.index)
    ax.plot(t, y, color=ACTUAL, lw=1, label="actual $Y_{t,h}$")
    ax.plot(t, f.fittedvalues, color=FITTED, lw=1, label="HAR fitted")
    ax.set_yscale("log")
    ax.set_title(f"h = {h}", loc="left", fontsize=10)
    ax.grid(axis="y", color="#e5e4e0", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, ncol=2, loc="upper left")
axes[1].set_ylabel("realized variance (log scale)")
fig.suptitle("In-sample HAR-RV fit on train (2016–2020)", x=0.01, ha="left")
plt.show()